# story_moe on Colab

Runs the same code that lives in the repo — this notebook only sets up and launches.
No model logic here, by design.

**Before running:** Runtime → Change runtime type → GPU. Pick the accelerator you
intend to use for **both** the dense and the MoE run and stay on it. Colab hands out
whatever is free, and a dense run on an A100 against an MoE run on a T4 is not a
comparison.

## 1. What GPU did Colab give us?

In [ ]:
import subprocess, torch

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip() or "no nvidia-smi")

print("torch", torch.__version__, "| built for CUDA", torch.version.cuda)
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime > Change runtime type > GPU, then rerun.")

NAME = torch.cuda.get_device_name(0)
BF16 = torch.cuda.is_bf16_supported()
PRECISION = "bf16" if BF16 else "fp16"
GPU_TAG = next((t for t in ("A100", "L4", "T4", "V100") if t in NAME), NAME.split()[-1])

print(f"device         : {NAME}")
print(f"bf16 supported : {BF16}")
print(f"--> use --precision {PRECISION} --require-gpu-name {GPU_TAG}")
print("    Record both in PROJECT_STATUS.md and reuse them for the MoE run.")

## 2. Get the code onto this machine

Pick **one**. Option A is worth the ten minutes of setup if you plan more than a
couple of Colab sessions — after it, every later update is one `git pull`.

In [ ]:
# --- Option A: GitHub (recommended) -------------------------------------------
# One-time, on your Windows machine, inside the repo folder:
#     git init
#     git add .
#     git commit -m "story_moe: days 1-3"
#     gh repo create story_moe --private --source=. --push
#         (or create an empty private repo on github.com, then:
#          git remote add origin https://github.com/<you>/story_moe.git
#          git branch -M main && git push -u origin main)
#
# Then here, every session:
REPO = "https://github.com/<your-username>/story_moe.git"   # <-- edit me

import os
if not os.path.exists("/content/story_moe"):
    !git clone -q $REPO /content/story_moe
else:
    !cd /content/story_moe && git pull -q
%cd /content/story_moe
!ls

In [ ]:
# --- Option B: Google Drive (no git) ------------------------------------------
# Copy the proj1 folder into your Google Drive (Drive desktop app, or drag it into
# drive.google.com) so it lands at MyDrive/story_moe. Then:
#
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/story_moe" /content/story_moe
# %cd /content/story_moe
#
# Re-copy after every change. That manual step is exactly what Option A removes.
#
# (For a one-off you can also zip the folder on Windows and use
#  `from google.colab import files; files.upload()`.)

## 3. Install dependencies

**Do not `pip install torch` here.** Colab ships a torch built against its own CUDA
driver; installing the PyPI wheel can replace it with one that does not match, and
you get a GPU that silently will not initialize. `--no-deps` installs this package
without letting `pyproject.toml` drag torch in behind your back.

In [ ]:
!pip install -q transformers datasets pyyaml pytest
!pip install -q -e . --no-deps

import torch
print("torch still", torch.__version__, "| cuda ok:", torch.cuda.is_available())

## 4. Mount Drive for anything you want to keep

The Colab VM's disk is deleted when the session ends, and sessions end on their own.
Checkpoints and the token cache go to Drive; everything else can stay local.

Setting `HF_HOME` to Drive is optional but saves re-downloading the TinyStories
parquet files (~1.8 GB) every session. It must be set *before* `datasets` is
imported, which is why it happens in this cell and not later.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/story_moe_artifacts"
CACHE = f"{BASE}/data_cache"
CKPT  = f"{BASE}/checkpoints"
os.makedirs(CACHE, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)

# Optional: persist the HuggingFace download cache across sessions.
os.environ["HF_HOME"] = f"{BASE}/hf_cache"

print("cache      :", CACHE)
print("checkpoints:", CKPT)

## 5. Tests, then data

In [ ]:
!pytest -q

In [ ]:
# Tokenize and pack. Skip if the Drive cache already has train.bin / validation.bin.
!python -m story_moe.data prepare --config configs/debug_dense.yaml --cache-dir "$CACHE"
!python -m story_moe.data show    --config configs/debug_dense.yaml --cache-dir "$CACHE"

## 6. Throughput probe

This is the measurement that decides the rest of the project: how large a training
subset to use, and whether the headline dense-vs-MoE comparison runs at the debug
tier or the 6-layer / d_model=384 tier. Read the `tok/s` column.

In [ ]:
!python -m story_moe.train train \
    --config configs/debug_dense.yaml \
    --data-cache "$CACHE" --out-dir "$CKPT/probe" \
    --precision $PRECISION --require-gpu-name $GPU_TAG \
    --max-updates 50

## 7. The baseline run

Set `--max-tokens` from the probe: `tokens = tok_per_s * seconds_you_are_willing_to_wait`.
Watch the `passes over data` line the trainer prints — above about 2.0 you are mostly
measuring memorization, and the fix is a larger `--train-stories`, not a longer run.

In [ ]:
# Example: a larger subset for a closer-to-single-pass run.
# !python -m story_moe.data prepare --config configs/debug_dense.yaml \
#     --cache-dir "$CACHE" --train-stories 22000 --val-stories 2000

!python -m story_moe.train train \
    --config configs/debug_dense.yaml \
    --data-cache "$CACHE" --out-dir "$CKPT/debug_dense" \
    --precision $PRECISION --require-gpu-name $GPU_TAG

## 8. If the session dies

It will. Reconnect, rerun cells 1–4, then resume from the checkpoint on Drive:

In [ ]:
!python -m story_moe.train train \
    --config configs/debug_dense.yaml \
    --data-cache "$CACHE" --out-dir "$CKPT/debug_dense" \
    --precision $PRECISION --require-gpu-name $GPU_TAG \
    --resume "$CKPT/debug_dense/latest.pt"

## 9. Bring the results back

`results/<name>_train.json` holds the full loss history, the parameter counts, the
processed/unique token counts, and the validation numbers. Copy it to Drive so it
survives, and paste the printed summary into the conversation.

In [ ]:
!mkdir -p "$BASE/results" && cp -v results/*.json "$BASE/results/" 2>/dev/null || echo "no results yet"